In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("hf://datasets/sharjeelyunus/github-issues-dataset/github_issues_dataset.parquet")

c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df.drop(["id", "repo"], axis=1, inplace=True)

## Normalize labels column

In [4]:
import re
import pandas as pd

def clean_labels(raw_labels):
    if pd.isna(raw_labels):
        return ["unknown"]
        
    noise_labels = {"p1", "p2", "p3", "p4", "p5"}
    parts = [l.strip().lower() for l in str(raw_labels).split(",")]
    meaningful = [
        l for l in parts
        if re.search(r"[a-z]", l) and l not in noise_labels
    ]
    return meaningful if meaningful else ["unknown"]

df["labels-list"] = df["labels"].apply(clean_labels)


In [5]:
from collections import Counter

all_labels = [l for row in df["labels-list"] for l in row]
counts = Counter(all_labels)

In [17]:
import json

with open("counts.json", "w") as f:
    json.dump(counts, f)

In [6]:
label_map = {
    "Bug": [
        "bug",
        "c-bug",
        "issue-bug",
        "type: bug",
        "bug-report",
        "kind/bug",
        "crash",
        "c: crash",
        "needsfix",
        "regression",
        "c: regression",
        "freeze-slow-crash-leak",
        "i-crash",
        "bug 🐛",
        "🤖:bug",
        "bug :beetle:",
        "s-bug-has-test",
        "i-ice",
        "bug-vim",
    ],
    "Feature Request": [
        "feature-request",
        "enhancement",
        "suggestion",
        "c: new feature",
        "c: proposal",
        "feature",
        "idea-enhancement",
        "feature request",
        "c-enhancement",
        "c-feature-request",
        "featurerequest",
        "type: feature request",
        "idea-new powertoy",
        "new feature",
        "function request",
        "feat",
        "💡 feature request",
        "kind/feature",
        "type:feature",
        "issue-feature",
        "improvement",
        "experience enhancement",
    ],
    "Performance": [
        "performance",
        "c: performance",
        "module: performance",
        "perf",
        "perf: speed",
        "perf: memory",
        "i-slow",
        "i-heavy",
        "i-compiletime",
        "freeze-slow-crash-leak",
        "module: memory usage",
        "usability",
    ],
    "Documentation": [
        "documentation",
        "docs",
        "doc",
        "module: docs",
        "d: api docs",
        "a-docs",
        "category: documentation",
        "area: docs",
        "addon: docs",
        "d: examples",
        "examples",
    ],
    "Support": [
        "question",
        "question / support",
        "question (invalid tracker)",
        "discussion",
        "asking-for-help-with-local-system-issues",
        "in discussion",
        "under-discussion",
        "site-support-request",
        "request",
    ],
    "Platform-specific": [
        "platform-android",
        "platform-ios",
        "platform-web",
        "platform-windows",
        "platform-linux",
        "platform-mac",
        "platform:windows",
        "platform:macos",
        "platform:android",
        "platform:ios",
        "platform:web",
        "platform:linuxbsd",
        "windows",
        "macos",
        "linux",
        "mobile",
        "os-windows",
        "o-windows",
    ],
    "Needs Triage": [
        "needs-triage",
        "needs triage",
        "triaged",
        "needsinvestigation",
        "needs investigation",
        "awaiting more feedback",
        "needs more info",
        "s-needs-repro",
        "needs-repro",
        "needs reproduction",
        "status: unconfirmed",
        "unconfirmed",
        "info-needed",
        "confirmation-pending",
        "in triage",
        "triage",
    ],
    "Refactoring": [
        "c: tech-debt",
        "c-cleanup",
        "c-optimization",
        "debt",
        "better-engineering",
        "polish",
        "build",
        "testing",
        "a: tests",
        "a-testsuite",
        "module: flaky-tests",
    ],
    "Security": [
        "nsfw",
        "geo-restricted",
        "geo-blocked",
    ],
    "Meta / Process (discard)": [
        "good first issue",
        "help wanted",
        "stale",
        "inactive",
        "wip",
        "rescheduled",
        "skipped",
        "auto-transferred",
        "has workaround",
        "workaround available",
        "patch-available",
        "fix available",
        "newer patch available",
        "waiting for 👍",
        "needs-team-response",
        "team",
        "oncall: pt2",
        "oncall: distributed",
        "oncall: jit",
        "oncall: export",
        "lifecycle/frozen",
        "stat:awaiting tensorflower",
        "asyncawait-triaged",
    ],
}

In [7]:
import random

alias_to_group = {}
for group, aliases in label_map.items():
    for alias in aliases:
        alias_to_group[alias.lower()] = group

discard_categories = {"Meta / Process (discard)"}

def process_labels(labels):
    mapped = []
    seen = set()
    group_counts = {}
    
    for label in labels:
        group = alias_to_group.get(label.lower(), "unknown")
        if group in discard_categories:
            continue
            
        group_counts[group] = group_counts.get(group, 0) + 1
        
        if group not in seen:
            mapped.append(group)
            seen.add(group)
    
    # Mapped array resolution
    if len(mapped) > 1 and "unknown" in mapped:
        mapped.remove("unknown")
    final_mapped = mapped if mapped else ["unknown"]
    
    # Atomic scalar resolution (fixing the 'unknown' dominance)
    if not group_counts:
        final_atomic = "unknown"
    else:
        if len(group_counts) > 1 and "unknown" in group_counts:
            del group_counts["unknown"]
        max_count = max(group_counts.values())
        tied_groups = [g for g, c in group_counts.items() if c == max_count]
        final_atomic = random.choice(tied_groups)
        
    return final_mapped, final_atomic

df["labels-mapped"], df["labels-atomic"] = zip(*df["labels-list"].apply(process_labels))
df["labels-mapped"].explode().value_counts().head(20)


labels-mapped
Bug                  33309
Needs Triage         27490
Feature Request      25570
unknown              23710
Platform-specific     7876
Support               4944
Documentation         4280
Performance           3806
Refactoring           3192
Security               370
Name: count, dtype: int64

In [8]:
target_share = 0.83
total = sum(counts.values())
target = target_share * total

running = 0
n = 0
for _, c in counts.most_common():
    running += c
    n += 1
    if running >= target:
        break

print("n =", n)
print("coverage =", running / total)

n = 528
coverage = 0.8300270540492855


In [9]:
df["labels-atomic"].value_counts()


labels-atomic
Bug                  29545
unknown              23710
Needs Triage         22186
Feature Request      21562
Platform-specific     5345
Support               4004
Documentation         2980
Performance           2235
Refactoring           2192
Security               314
Name: count, dtype: int64

In [10]:
df_clean = df.drop(["labels", "labels-list", "labels-mapped"], axis=1)

In [11]:
X = df_clean.drop(['priority', 'severity', 'labels-atomic'], axis=1)
y = df_clean["labels-atomic"]

In [12]:
X["text"] = X["title"].fillna("") + " [SEP] " + X["body"].fillna("")
X.drop(["title", "body"], axis=1, inplace=True)

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

models = {
    "logreg" : LogisticRegression(max_iter=3000, class_weight="balanced"),
    "svm" : LinearSVC(class_weight="balanced")
}

results = []

for name, clf in models.items():
    pipe = Pipeline([
        ("tdif", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1,2),
        min_df=2, 
        max_features=50000,
        sublinear_tf=True)),

        ("clf", clf)
    ])

    pipe.fit(X_train["text"], y_train)
    y_pred = pipe.predict(X_test["text"])

    acc_score = accuracy_score(y_test, y_pred)
    f1_val = f1_score(y_test, y_pred, average='macro')

    results.append((name, acc_score, f1_val))
    print(f"\n{name}")
    print("acc_scoreuracy:", acc_score)
    print("macro f1_val:", f1_val)
    from sklearn.metrics import classification_report
    print(classification_report(y_test, y_pred))



logreg
acc_scoreuracy: 0.6024983563445102
macro f1_val: 0.4928809409418326
                   precision    recall  f1-score   support

              Bug       0.75      0.64      0.69      5909
    Documentation       0.32      0.58      0.41       596
  Feature Request       0.66      0.59      0.62      4313
     Needs Triage       0.72      0.68      0.70      4437
      Performance       0.29      0.57      0.38       447
Platform-specific       0.45      0.71      0.55      1069
      Refactoring       0.25      0.58      0.35       438
         Security       0.17      0.84      0.29        63
          Support       0.29      0.47      0.36       801
          unknown       0.69      0.50      0.58      4742

         accuracy                           0.60     22815
        macro avg       0.46      0.62      0.49     22815
     weighted avg       0.65      0.60      0.62     22815


svm
acc_scoreuracy: 0.6283147052377822
macro f1_val: 0.4959037662336252
                   pre